# 03 — DL Experiments
Single asset. Set TICKER, MODEL_NAME, and MODE in the configuration cell below. The GPU queue (`run_dl_queue.py`) runs the same pipeline automatically for all 300 combinations.

In [ ]:
# ─── CONFIGURATION ────────────────────────────────────────
TICKER      = "PETR4.SA"          # any ticker from TICKERS list
MODEL_NAME  = "CNN"               # CNN | LSTM | CNN_LSTM | Transformer
MODE        = "raw"               # raw | db4 | learned_wavelet
# Override specific hyperparameters (leave empty {} for defaults)
CONFIG_OVERRIDE = {}
# ──────────────────────────────────────────────────────────

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../..").resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config.experiment_config import DL_TRAINING_CONFIG, DL_MODELS_CONFIG
from src.pipeline import FinancialExperimentPipeline
from src.data_loader import load_processed, load_labels
from src.evaluation import ResultsManager, ClassificationEvaluator, FinancialMetrics
from src.backtest import simulate_strategy, buy_and_hold_returns, cumulative_returns

RESULTS_DIR = Path("results")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print(f"Experiment: {TICKER} | {MODEL_NAME} | {MODE}")

## 1. Data Preview

In [ ]:
features = load_processed(TICKER)
labels   = load_labels(TICKER)
common   = features.index.intersection(labels.index)
features = features.loc[common].dropna()
labels   = labels.loc[features.index]

print(f"Features: {features.shape}")
print(f"Labels: {labels.shape}  |  classes: {labels.value_counts().sort_index().to_dict()}")
print(f"Date range: {features.index[0].date()} \u2192 {features.index[-1].date()}")
features.describe().round(4)

## 2. Run Pipeline

In [ ]:
pipeline = FinancialExperimentPipeline(
    ticker=TICKER,
    model_name=MODEL_NAME,
    mode=MODE,
    config=CONFIG_OVERRIDE,
    results_dir=str(RESULTS_DIR),
)

print(f"Results path: {pipeline.job_results_dir}")
print(f"Already done: {pipeline.is_done()}")

metrics = pipeline.run()
print("\n=== Experiment Complete ===")

## 3. Results

In [ ]:
print("\u2500\u2500 ML Metrics \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
ml_keys = [k for k in metrics if not k.startswith("fin_")]
for k in ml_keys:
    print(f"  {k:<30} {metrics[k]:.4f}" if isinstance(metrics[k], float) else f"  {k}: {metrics[k]}")

print("\n\u2500\u2500 Financial Metrics \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500")
fin_keys = [k for k in metrics if k.startswith("fin_")]
for k in fin_keys:
    print(f"  {k:<30} {metrics[k]:.4f}" if isinstance(metrics[k], float) else f"  {k}: {metrics[k]}")

## 4. Load Predictions & Visualise

In [ ]:
import glob

pred_files = sorted((RESULTS_DIR / TICKER / f"{MODEL_NAME}_{MODE}").glob("predictions_fold*.npz"))
y_true_all, y_pred_all = [], []
for pf in pred_files:
    data = np.load(pf)
    y_true_all.append(data["y_true"])
    y_pred_all.append(data["y_pred"])

y_true = np.concatenate(y_true_all) if y_true_all else np.array([])
y_pred = np.concatenate(y_pred_all) if y_pred_all else np.array([])

print(f"Total test samples: {len(y_true)}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

if len(y_true):
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["sell", "hold", "buy"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix \u2014 {TICKER} / {MODEL_NAME} / {MODE}")
    plt.tight_layout()
    plt.show()

## 5. Learned Filter Analysis (if mode=learned_wavelet)

In [ ]:
if MODE == "learned_wavelet":
    import glob
    model_files = sorted((Path("saved_models") / TICKER / f"{MODEL_NAME}_{MODE}").glob("fold_*.keras"))
    if model_files:
        import tensorflow as tf
        model = tf.keras.models.load_model(str(model_files[-1]), compile=False)
        wavelet_layers = [l for l in model.layers if "learned_wavelet" in l.name.lower()]
        if wavelet_layers:
            wl = wavelet_layers[0]
            print(f"Wavelet layer: {wl.name}")
            # Plot learned filter frequency response
            # (implementation depends on exact layer API)
            print("Wavelet layer found. Filter visualisation requires accessing layer weights.")
        else:
            print("No wavelet layer found in loaded model.")
    else:
        print("No saved model found. Run the pipeline first.")
else:
    print(f"Filter analysis only available for mode='learned_wavelet' (current: {MODE!r})")